In [1]:
import scanpy as sc
import pandas as pd

In [2]:
adata = sc.read_h5ad("raw/SrivatsanTrapnell2020_sciplex3.h5ad")

In [3]:
adata

AnnData object with n_obs × n_vars = 799317 × 110983
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type'
    var: 'ensembl_id'

In [15]:
adata.layers["counts"] = adata.X.copy()

In [16]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [17]:
sc.pp.highly_variable_genes(adata,n_top_genes=2000)
adata = adata[:, adata.var['highly_variable']]

In [18]:
adata.write_h5ad(f"raw/Sciplex3_hvg2000.h5ad")

In [46]:
adata = sc.read_h5ad("raw/Sciplex3_hvg2000.h5ad")

In [57]:
adata = adata[adata.obs[["time", "perturbation","dose_value"]].notna().all(axis=1), :].copy() #部分time是nan

In [54]:
adata_24 = adata[adata.obs["time"]==24].copy()

In [55]:
adata_24

AnnData object with n_obs × n_vars = 680685 × 2000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type'
    var: 'ensembl_id', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p'
    layers: 'counts'

In [59]:
adata_24.write_h5ad(f"raw/Sciplex3_24h_hvg2000.h5ad")

In [51]:
adata_72 = adata[adata.obs["time"]==72].copy()

In [52]:
adata_72

AnnData object with n_obs × n_vars = 82110 × 2000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type'
    var: 'ensembl_id', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p'
    layers: 'counts'

In [60]:
adata_72.write_h5ad(f"raw/Sciplex3_72h_hvg2000.h5ad")

In [56]:
adata.obs["time"].value_counts()

time
24.0    680685
72.0     82110
Name: count, dtype: int64

In [28]:
adata

AnnData object with n_obs × n_vars = 799317 × 2000
    obs: 'ncounts', 'well', 'plate', 'cell_line', 'replicate', 'time', 'dose_value', 'pathway_level_1', 'pathway_level_2', 'perturbation', 'target', 'pathway', 'dose_unit', 'celltype', 'disease', 'cancer', 'tissue_type', 'organism', 'perturbation_type'
    var: 'ensembl_id', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p'
    layers: 'counts'

In [39]:
adata.obs["time"].value_counts()

time
24.0    680685
72.0     82110
Name: count, dtype: int64

In [64]:
import pandas as pd

obs = adata_24.obs.copy()

# 定义唯一孔：plate + well
obs["plate_well"] = obs["plate"].astype(str) + "_" + obs["well"].astype(str)

# 统计每个 (dose_value, perturbation, time) 对应多少个唯一孔
condition_well_counts = (
    obs.groupby(["dose_value", "perturbation", "time"])["plate_well"]
    .nunique()
    .reset_index(name="n_wells")
    .sort_values(["dose_value", "perturbation", "time"])
)

condition_well_counts = condition_well_counts[condition_well_counts["n_wells"] != 0]

print(condition_well_counts)


     dose_value                  perturbation  time  n_wells
188         0.0                       control  24.0       96
189        10.0  2-Methoxyestradiol (2-MeOE2)  24.0        6
190        10.0                       (+)-JQ1  24.0        6
191        10.0                         A-366  24.0        6
192        10.0                       ABT-737  24.0        6
..          ...                           ...   ...      ...
939     10000.0                        WP1066  24.0        6
940     10000.0                       XAV-939  24.0        6
941     10000.0  YM155 (Sepantronium Bromide)  24.0        6
942     10000.0                     ZM 447439  24.0        6
943     10000.0                      Zileuton  24.0        6

[753 rows x 4 columns]


/tmp/ipykernel_2908227/4262689484.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  obs.groupby(["dose_value", "perturbation", "time"])["plate_well"]


In [65]:
adata_24.obs["perturbation"].value_counts()

perturbation
control                                15494
Baricitinib (LY3009104, INCB028050)     4315
Tranylcypromine (2-PCPA) HCl            4291
WP1066                                  4263
RG108                                   4244
                                       ...  
Alvespimycin (17-DMAG) HCl              2089
Patupilone (EPO906, Epothilone B)       1822
Flavopiridol HCl                        1729
Epothilone A                            1426
YM155 (Sepantronium Bromide)            1007
Name: count, Length: 189, dtype: int64

In [66]:
adata_72.obs["perturbation"].value_counts()

perturbation
Iniparib (BSI-201)                              2433
Ellagic acid                                    2375
BMS-911543                                      2323
Resveratrol                                     2286
TMP195                                          2253
Divalproex Sodium                               2221
PFI-1 (PF-6405761)                              2201
Ruxolitinib (INCB018424)                        2198
UNC1999                                         2174
MC1568                                          2173
G007-LK                                         2168
(+)-JQ1                                         2144
Roxadustat (FG-4592)                            2127
SRT3025 HCl                                     2107
FLLL32                                          2100
Fasudil (HA-1077) HCl                           2099
Decitabine                                      2091
control                                         2084
Azacitidine                      

In [67]:
obs = adata_24.obs.copy()
obs["plate_well"] = obs["plate"].astype(str) + "_" + obs["well"].astype(str)

wells_per_perturb = (
    obs.groupby("perturbation", observed=True)["plate_well"]
    .nunique()
    .sort_values(ascending=False)
)

ratio = wells_per_perturb["control"] / wells_per_perturb[wells_per_perturb.index != "control"].iloc[0]
print(ratio)


4.0


In [43]:
adata.obs["well"].value_counts()

well
plate10_B2     1428
plate10_C12    1428
plate10_B3     1418
plate10_H5     1391
plate10_B8     1388
               ... 
plate2_E8        14
plate2_D6        12
plate2_D2        12
plate2_E2         8
plate2_C1         7
Name: count, Length: 864, dtype: int64

In [26]:
adata.obs["cell_line"].value_counts()

cell_line
MCF7    344862
A549    244281
K562    173652
Name: count, dtype: int64

In [19]:
adata.obs['dose_value'].value_counts()

dose_value
10.0       202725
100.0      192858
1000.0     183356
10000.0    166278
0.0         17578
Name: count, dtype: int64

In [20]:
adata.obs["perturbation"].value_counts()

perturbation
control                              17578
Ellagic acid                          6257
Divalproex Sodium                     6203
Ruxolitinib (INCB018424)              6143
MC1568                                6126
                                     ...  
Alvespimycin (17-DMAG) HCl            2089
Patupilone (EPO906, Epothilone B)     1822
Flavopiridol HCl                      1729
Epothilone A                          1426
YM155 (Sepantronium Bromide)          1007
Name: count, Length: 189, dtype: int64

In [22]:
df = pd.DataFrame(
    adata.obs["perturbation"].unique(),
    columns=["drug"]
)
df.to_csv("target_drugs_sciplex3.csv", index=False)

In [25]:
adata.obs["time"].value_counts()

time
24.0    680685
72.0     82110
Name: count, dtype: int64